# Coalescent Constraints Using Bipartitions

This notebook demonstrates how to use bipartitions extracted from phylogenetic trees to constrain coalescent simulations. The key idea is that lineages can only coalesce if all their descendants belong to the same partition.

In [ ]:
from coalescent_constraints import (
    Bipartition,
    parse_newick,
    bipartitions_from_tree,
    Lineage,
    ConstraintSet,
    PartitionRefinement,
    CoalescentState,
)

## 1. Parsing a Newick Tree

We start by parsing a simple tree with 4 samples. The tree `((0,1),(2,3))` represents:
- Samples 0 and 1 are sister taxa
- Samples 2 and 3 are sister taxa
- These two pairs are joined at the root

In [ ]:
newick = "((0,1),(2,3));"
tree = parse_newick(newick)

print(f"Root has {len(tree.children)} children")
print(f"All leaf indices: {tree.get_leaf_set()}")
for i, child in enumerate(tree.children):
    print(f"  Child {i} descendants: {child.get_leaf_set()}")

## 2. Extracting Bipartitions

Each internal node in the tree defines a bipartition: its descendants vs. the rest. For our tree, the main bipartition is `{0,1} | {2,3}`.

In [ ]:
n_samples = 4
bipartitions = bipartitions_from_tree(tree, n_samples)

print(f"Extracted {len(bipartitions)} bipartition(s):")
for bp in bipartitions:
    print(f"  {bp}")

## 3. Understanding Bipartitions

A bipartition divides samples into two groups. We can check if two samples are in the same partition:

In [ ]:
bp = bipartitions[0]
print(f"Bipartition: {bp}")
print()
print("Pairwise partition membership:")
for i in range(n_samples):
    for j in range(i + 1, n_samples):
        same = bp.same_partition(i, j)
        print(f"  Samples {i} and {j}: {'same partition' if same else 'different partitions'}")

## 4. Coalescent State and Constraints

Now let's set up a coalescent simulation with constraints. Initially, each sample is its own lineage.

In [ ]:
state = CoalescentState(n_samples)
print(f"Initial state: {state}")
print(f"Active lineages: {state.lineages}")

Without constraints, any pair of lineages can coalesce:

In [ ]:
pairs = state.get_compatible_pairs()
print(f"Compatible pairs (no constraints): {len(pairs)} pairs")
for a, b in pairs:
    print(f"  {a} + {b}")

Now let's add the bipartition constraint:

In [ ]:
for bp in bipartitions:
    state.add_constraint(bp)

print(f"State with constraints: {state}")
pairs = state.get_compatible_pairs()
print(f"\nCompatible pairs (with constraint): {len(pairs)} pairs")
for a, b in pairs:
    print(f"  {a} + {b}")

The constraint restricts which lineages can coalesce: only samples in the same partition can merge.

## 5. Performing Coalescence

Let's coalesce samples 0 and 1:

In [ ]:
# Find lineages for samples 0 and 1
lin0 = next(l for l in state.lineages if 0 in l.descendant_indices())
lin1 = next(l for l in state.lineages if 1 in l.descendant_indices())

print(f"Coalescing {lin0} and {lin1}")
merged_01 = state.coalesce(lin0, lin1)
print(f"Result: {merged_01}")
print(f"\nState after coalescence: {state}")
print(f"Active lineages: {state.lineages}")

Now check what coalescences are still possible:

In [ ]:
pairs = state.get_compatible_pairs()
print(f"Compatible pairs: {len(pairs)}")
for a, b in pairs:
    print(f"  {a} + {b}")

# Note: the merged lineage {0,1} cannot coalesce with 2 or 3
# because they are in different partitions

Coalesce samples 2 and 3:

In [ ]:
lin2 = next(l for l in state.lineages if 2 in l.descendant_indices())
lin3 = next(l for l in state.lineages if 3 in l.descendant_indices())

merged_23 = state.coalesce(lin2, lin3)
print(f"Coalesced to: {merged_23}")
print(f"\nState: {state}")
print(f"Active lineages: {state.lineages}")

## 6. Auto-updating Constraints

The constraint system automatically removes bipartitions when they are satisfied. A bipartition is satisfied when each partition has been fully coalesced into a single lineage.

After coalescing samples 0+1 and 2+3, the bipartition `{0,1} | {2,3}` is satisfied:

In [ ]:
print(f"Constraints remaining: {len(state.constraints)}")
pairs = state.get_compatible_pairs()
print(f"Compatible pairs: {len(pairs)}")
print(f"Can coalesce: {state.constraints.can_coalesce(merged_01, merged_23)}")

The constraint was automatically removed! This means we can proceed to the final coalescence without manual intervention:

In [ ]:
# Final coalescence - no need to clear constraints manually!
mrca = state.coalesce(merged_01, merged_23)
print(f"MRCA lineage: {mrca}")
print(f"Coalescence complete: {state.is_mrca()}")

## 7. Partition Refinement

For efficiency with many samples, `PartitionRefinement` groups samples by their signature across all bipartitions:

In [ ]:
# A more complex tree
tree2 = parse_newick("(((0,1),2),(3,4));")
bps2 = bipartitions_from_tree(tree2, 5)

print("Bipartitions from (((0,1),2),(3,4)):")
for bp in bps2:
    print(f"  {bp}")

# Use partition refinement to find groups
refinement = PartitionRefinement(bps2, 5)
print(f"\nRefined groups (samples that can coalesce):")
for group in refinement.groups:
    print(f"  {group}")

## 8. Working Example: Constrained Coalescent Simulation

Here's a complete example showing how constraints from a tree guide coalescence:

In [ ]:
import random

def simulate_constrained_coalescent(newick_str, n_samples, seed=42):
    """Simulate coalescence respecting tree-derived constraints."""
    random.seed(seed)
    
    # Parse tree and extract bipartitions
    tree = parse_newick(newick_str)
    bipartitions = bipartitions_from_tree(tree, n_samples)
    
    # Initialize state with constraints
    state = CoalescentState(n_samples)
    for bp in bipartitions:
        state.add_constraint(bp)
    
    print(f"Tree: {newick_str}")
    print(f"Initial constraints: {len(state.constraints)} bipartition(s)")
    print(f"Initial lineages: {len(state.lineages)}")
    print()
    
    step = 0
    while not state.is_mrca():
        pairs = state.get_compatible_pairs()
        
        if not pairs:
            # This shouldn't happen with auto-updating constraints
            print(f"Step {step}: ERROR - No compatible pairs!")
            break
        
        # Randomly choose a pair to coalesce
        lin_a, lin_b = random.choice(pairs)
        merged = state.coalesce(lin_a, lin_b)
        
        print(f"Step {step}: {lin_a} + {lin_b} -> {merged}")
        print(f"         Remaining constraints: {len(state.constraints)}")
        step += 1
    
    print(f"\nCoalescence complete in {step} steps")
    return state

# Run simulation - constraints auto-update as bipartitions are satisfied!
simulate_constrained_coalescent("((0,1),(2,3));", 4)

In [ ]:
# A larger example
simulate_constrained_coalescent("(((0,1),(2,3)),((4,5),(6,7)));", 8)